In [1]:
import os
import matplotlib.pyplot as plt
import pandas as pd

def load_data(folder_path):
    reviews = []
    labels = []
    for label in ['pos', 'neg']:
        dir_path = os.path.join(folder_path, label)
        for filename in os.listdir(dir_path):
            filepath = os.path.join(dir_path, filename)
            with open(filepath, 'r', encoding='utf-8') as f:
                reviews.append(f.read())
            labels.append(1 if label == 'pos' else 0)

    df = pd.DataFrame({'review': reviews, 'label': labels})
    return df

train_df = load_data('../data/IMDB/train')
test_df  = load_data('../data/IMDB/test')

train_df.head()


,review,label
0,"This isn't the comedic Robin Williams, nor is ...",1
1,Yes its an art... to successfully make a slow ...,1
2,I absolutely LOVED this film! I do not at all ...,1
3,Somewhat funny and well-paced action thriller ...,1
4,"Sure, Titanic was a good movie, the first time...",1


# Text Preprocessing

In [2]:
import string

def preprocess_text(text):
    text = text.lower()
    text = text.replace("<br />", " ")
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    return tokens

train_df['tokens'] = train_df['review'].apply(preprocess_text)
test_df['tokens'] = test_df['review'].apply(preprocess_text)

train_df.head()

,review,label,tokens
0,"This isn't the comedic Robin Williams, nor is ...",1,"[this, isnt, the, comedic, robin, williams, no..."
1,Yes its an art... to successfully make a slow ...,1,"[yes, its, an, art, to, successfully, make, a,..."
2,I absolutely LOVED this film! I do not at all ...,1,"[i, absolutely, loved, this, film, i, do, not,..."
3,Somewhat funny and well-paced action thriller ...,1,"[somewhat, funny, and, wellpaced, action, thri..."
4,"Sure, Titanic was a good movie, the first time...",1,"[sure, titanic, was, a, good, movie, the, firs..."


# Build Vocabulary

In [3]:
def build_vocabulary(tokens_series):
    vocab = {}
    index = 0
    for tokens in tokens_series:
        for word in tokens:
            if word not in vocab:
                vocab[word] = index
                index += 1
    return vocab

vocab = build_vocabulary(train_df['tokens'])
print(f"Vocabulary size: {len(vocab)}")

Vocabulary size: 49870


# Bag of Words

In [4]:
import numpy as np

def build_bow_matrix(tokens_series, vocab):
    tokens_list = list(tokens_series)
    matrix = np.zeros((len(tokens_list), len(vocab)), dtype=np.int16)
    for i, tokens in enumerate(tokens_list):
        for word in tokens:
            if word in vocab:
                matrix[i][vocab[word]] += 1
    return matrix


train_bow = build_bow_matrix(train_df['tokens'], vocab)
test_bow  = build_bow_matrix(test_df['tokens'], vocab)

# Class Priors

In [5]:
def compute_prior(df):
    total     = len(df)
    prior_pos = len(df[df['label'] == 1]) / total
    prior_neg = len(df[df['label'] == 0]) / total
    return prior_pos, prior_neg

prior_pos, prior_neg = compute_prior(train_df)
print(f"P(positive) = {prior_pos}")
print(f"P(negative) = {prior_neg}")

P(positive) = 0.4939892731644165
P(negative) = 0.5060107268355835


In [20]:
def compute_word_prob(labels, vocab, train_bow):
    word_prob = {}

    for cls in set(labels):
        cls_indices = []
        for i in range(len(labels)):
            if labels[i] == cls:
                cls_indices.append(i)

        cls_bow = train_bow[cls_indices]
        word_count = cls_bow.sum(axis=0)
        total_words = word_count.sum()
        word_prob[cls] = (word_count + 1) / (total_words + len(vocab))

    return word_prob